In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from Observer_Manager import ObserverManager
from BEMT_Acoustic_Job import Job

In [ ]:
# Step 1: Generate points on a sphere
def sample_points_on_sphere(radius, num_points):
    positions = []
    for _ in range(num_points):
        theta = np.random.uniform(0, 2 * np.pi)  # Azimuthal angle
        phi = np.arccos(np.random.uniform(-1, 1))  # Polar angle
        x = radius * np.sin(phi) * np.cos(theta)
        y = radius * np.sin(phi) * np.sin(theta)
        z = radius * np.cos(phi)
        positions.append([x, y, z])
    return np.array(positions)

def sample_points_on_circles(radius, x_positions, points_per_circle):
    positions = []

    for x_position in x_positions:
        if abs(x_position) > radius:
            raise ValueError("x_position must be within the range of the hemisphere's radius.")

        # Compute the radius of the circle in the y-z plane at the given x position
        circle_radius = np.sqrt(radius**2 - x_position**2)

        # Sample points around the circle in the y-z plane
        for phi in np.linspace(0, 2 * np.pi, points_per_circle, endpoint=False):
            y = circle_radius * np.cos(phi)
            z = circle_radius * np.sin(phi)
            positions.append([x_position, y, z])

    return np.array(positions)

# Parameters
radius = 1.88
num_points = 10
x_pos = [-1.5, -1, -0.5, 0, 0.5, 1, 1.5]
position = sample_points_on_circles(radius, x_pos, num_points)
#position = sample_points_on_sphere(radius, num_points)


In [ ]:
"""DEFINE OBSERVER OBJECTS"""
observer_manager = ObserverManager.from_positions(position)

In [ ]:
observer_manager.plot_observer_positions()

In [ ]:
"""CREATE ANALYSIS OBJECT"""
propeller_name = "10x7E"
RPM = 7000
v_inf = 0 
revolutions = 1 #revolutions to simulate

analysis = Job(propeller_name=propeller_name, RPM=RPM, v_inf=v_inf, revolutions=revolutions, observer_manager=observer_manager)

In [ ]:
"""RUN BEMT"""
analysis.run_BEMT()

In [ ]:
"""VIEW BEMT SOLUTION DATA"""
analysis.bemt_analysis.solution_data

In [ ]:
"""RUN ACOUSTIC ANALYSIS"""
analysis.run_acoustic_analysis()

In [ ]:
OSPL_list = []
for observer in analysis.observer_manager:
    OSPL_list.append(observer.OSPL)

In [ ]:
OSPL_list

In [ ]:
time = analysis.observer_manager.observers[0].time_matrix
pressure_m = analysis.observer_manager.observers[0].pressure_matrix_m
pressure_d = analysis.observer_manager.observers[0].pressure_matrix_d
t_common = analysis.observer_manager.observers[0].t_com

In [ ]:
for i in [0,1]:
    plt.plot(time[:,i], pressure_m[:,i], label=i, marker='.')
plt.grid()
plt.legend()
plt.show()

In [ ]:
import numpy as np 
from scipy.interpolate import InterpolatedUnivariateSpline

n_sources_tot = time.shape[1]
p_m_interp = np.zeros((len(t_common), n_sources_tot))
p_d_interp = np.zeros((len(t_common), n_sources_tot))
for source in range(0, n_sources_tot):
    spl_1 = InterpolatedUnivariateSpline(time[:, source], pressure_m[:, source], k=3)
    p_m_interp[:,source] = spl_1(t_common)
    spl_2 = InterpolatedUnivariateSpline(time[:, source], pressure_d[:, source], k=3)
    p_d_interp[:,source] = spl_2(t_common)

In [ ]:
for i in range(0,4):
    plt.plot(t_common, p_m_interp[:,i], label=i, marker='.')
plt.grid()
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull

# Generate points and compute values
values = OSPL_list

# Extract x, y, z for plotting
x, y, z = position[:, 0], position[:, 1], position[:, 2]

# Plot the sphere with values as colors
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Use scatter plot for visualization
scatter = ax.scatter(x, y, z, c=values, cmap='viridis')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
# Add a color bar
colorbar = fig.colorbar(scatter, ax=ax, shrink=0.5, aspect=10, label='Values')

# Set axis properties
ax.set_box_aspect([1, 1, 1])  # Equal aspect ratio
ax.set_title("Values Mapped to Sphere")
plt.show()

In [ ]:
"""VIEW ACOUSTIC SOLUTION DATA"""
observer = analysis.observer_manager[1]

%matplotlib inline
fig, axs = plt.subplots(1,2)
fig.set_figheight(7)
fig.set_figwidth(15)
fontsize=10
labelsize=10

axs[0].plot(observer.t, observer.p_m, marker='.', label='monopole pressure')
axs[0].plot(observer.t, observer.p_d, marker='.', label='dipole pressure')
axs[0].plot(observer.t, observer.p_m+observer.p_d, marker='.', label='total pressure')
axs[0].set_xlabel('Time [s]', fontsize=fontsize)
axs[0].set_ylabel('Acoustic pressure [Pa]', fontsize=fontsize)
axs[0].grid(True)
axs[0].legend()

axs[1].semilogx(observer.frequency, observer.SPL, marker='.', label=f'OSPL: {observer.OSPL:.2f} dB')
axs[1].set_xlabel('Frequency [Hz]', fontsize=fontsize)
axs[1].set_ylabel('SPL [dB]', fontsize=fontsize)
axs[1].grid(True)
axs[1].legend()